# CUM Scale Test + Generalization

**Part 1 — Scale test:** Muon vs Combined at ~124M params (GPT-2 Small)
- WikiText-103, GPT-2 BPE tokenization
- Does combined's -0.016 advantage hold at 100x scale?

**Part 2 — Generalization:** Adam vs SmoothedAdam on MicroGPT (1.2M)
- Does temporal averaging of update directions help non-NS optimizers?

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tiktoken'], check=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ── Download & tokenize WikiText-103 ──
import tiktoken

DATA_DIR = 'benchmarks/tier0/data'
WIKI_PATH = os.path.join(DATA_DIR, 'wikitext103_tokens.pt')

if os.path.exists(WIKI_PATH):
    print('Loading cached tokens...')
    all_tokens = torch.load(WIKI_PATH)
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Downloading WikiText-103...')
    from datasets import load_dataset
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train')
    text = '\n\n'.join([t for t in ds['text'] if t.strip()])
    print(f'Text: {len(text)/1e6:.1f}M chars')
    
    print('Tokenizing with GPT-2 BPE...')
    enc = tiktoken.get_encoding('gpt2')
    token_list = enc.encode(text)
    all_tokens = torch.tensor(token_list, dtype=torch.long)
    torch.save(all_tokens, WIKI_PATH)

print(f'Tokens: {len(all_tokens)/1e6:.1f}M')
VOCAB_SIZE = 50257

In [ ]:
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize, newton_schulz_multi_resolution
from cum.utils import aspect_ratio_scale
from cum import CUM8v1, SmoothedAdam
print('Imports OK')

In [ ]:
# ── Model configs ──
CFG_100M = dict(d_model=768, n_heads=12, n_layers=12, d_ff=3072, ctx_len=256)
CFG_SMALL = dict(d_model=128, n_heads=4, n_layers=4, d_ff=512, ctx_len=256)

SEED = 42

_m = MicroGPT(vocab_size=VOCAB_SIZE, **CFG_100M)
n_total = sum(p.numel() for p in _m.parameters())
n_hidden = sum(p.numel() for p in _m.get_hidden_2d_params())
shapes = set(tuple(p.shape) for p in _m.get_hidden_2d_params())
print(f'100M model: {n_total/1e6:.1f}M total, {n_hidden/1e6:.1f}M hidden 2D')
print(f'Hidden shapes: {shapes}')
del _m

In [ ]:
# ── Datasets ──

class TokenDataset:
    def __init__(self, tokens, ctx_len, device, val_frac=0.01):
        self.ctx_len = ctx_len
        self.vocab_size = VOCAB_SIZE
        n = int(len(tokens) * (1 - val_frac))
        self.train_data = tokens[:n].to(device)
        self.val_data = tokens[n:].to(device)
        print(f'Train: {len(self.train_data)/1e6:.1f}M, Val: {len(self.val_data)/1e6:.1f}M tokens')

    def get_batch(self, batch_size, split='train'):
        data = self.train_data if split == 'train' else self.val_data
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


class CharDataset:
    def __init__(self, text, ctx_len, device):
        chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(chars)}
        self.vocab_size = len(chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train'):
        n = int(len(self.data) * 0.9)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


ds_100m = TokenDataset(all_tokens, ctx_len=CFG_100M['ctx_len'], device=DEVICE)

In [ ]:
# ── Training Infrastructure ──

class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def train_one(name, main_opt, aux_opt, model, dataset,
              total_steps=2000, batch_size=32, warmup_steps=200,
              base_lr=0.02, eval_every=250):
    model.train()
    trajectory = []
    sole_optimizer = main_opt is None
    if sole_optimizer:
        _aux_base_lr = aux_opt.param_groups[0]['lr']

    # Warmup torch.compile
    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        muon_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        if main_opt:
            for g in main_opt.param_groups:
                g['lr'] = muon_lr
        if sole_optimizer:
            aux_lr = get_lr(step, total_steps, warmup_steps, _aux_base_lr)
            for g in aux_opt.param_groups:
                g['lr'] = aux_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()
        loss.backward()
        if main_opt: main_opt.step()
        aux_opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training infra ready')

In [ ]:
# ── Factory + Runner ──

def make_model_and_opts(dataset, cfg, model_cfg, base_lr=0.02):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **model_cfg).to(DEVICE)
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception as e:
        print(f'torch.compile failed ({e}), continuing without')

    t = cfg['type']

    # Full-model optimizers (no Muon split)
    if t in ('AdamW_full', 'SmoothedAdam_full'):
        all_params = list(model.parameters())
        if t == 'AdamW_full':
            aux = torch.optim.AdamW(all_params, lr=cfg.get('lr', 1e-3),
                                    weight_decay=cfg.get('wd', 0.01))
        else:
            aux = SmoothedAdam(all_params, lr=cfg.get('lr', 1e-3),
                              weight_decay=cfg.get('wd', 0.01),
                              smooth_beta=cfg.get('smooth_beta', 0.5),
                              smooth_alpha=cfg.get('smooth_alpha', 0.15))
        return model, None, aux

    # Muon/CUM: split hidden 2D vs rest
    hidden_2d = model.get_hidden_2d_params()
    other = model.get_other_params()
    aux = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)

    if t == 'Muon':
        main = SimpleMuon(hidden_2d, lr=base_lr, ns_steps=5)
    elif t == '8v1':
        main = CUM8v1(
            hidden_2d, lr=base_lr,
            method=cfg.get('method', 'matrix'),
            mode=cfg.get('mode', 'combined'),
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
            total_steps=cfg.get('total_steps', 2000),
        )
    else:
        raise ValueError(f'Unknown: {t}')

    return model, main, aux


def run_all(configs, dataset, model_cfg, total_steps=2000, batch_size=32,
            warmup_steps=200, base_lr=0.02, eval_every=250):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, main_opt, aux_opt = make_model_and_opts(
                dataset, cfg, model_cfg, base_lr)
            val_loss, traj, elapsed = train_one(
                name, main_opt, aux_opt, model, dataset,
                total_steps, batch_size, warmup_steps, base_lr, eval_every)
            results.append(dict(name=name, val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='scale'):
    ref = results[0]['val_loss'] if results else None
    print(f'\n## {series} Results\n')
    print(f'| Config | Val Loss | vs Baseline | Time |')
    print(f'|--------|----------|-------------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r.get('error'):
            print(f'| {r["name"]} | FAILED | -- | {r["error"][:40]} |')
            continue
        vb = f'{r["val_loss"] - ref:+.4f}' if ref else '--'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vb} | {r["elapsed"]:.0f}s |')


print('Factory ready')

---
## Part 1: Scale Test — Muon vs Combined at 124M

- Model: GPT-2 Small (124M total, 85M hidden 2D)
- Data: WikiText-103 (~100M BPE tokens)
- 2000 steps, batch=32, ctx=256, lr=0.02
- Question: does combined's -0.016 advantage grow or shrink at 100x scale?

In [ ]:
CONFIGS_SCALE = [
    {'name': 'Muon NS=5', 'type': 'Muon'},
    {'name': 'CUM combined', 'type': '8v1', 'method': 'matrix', 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
]

results_scale = run_all(CONFIGS_SCALE, ds_100m, CFG_100M,
                        total_steps=2000, batch_size=32, warmup_steps=200,
                        base_lr=0.02, eval_every=250)
show_results(results_scale, 'Scale Test (124M params, WikiText-103)')

---
## Part 2: Generalization — Does Temporal Averaging Help Adam?

SmoothedAdam = AdamW + EMA of update directions (same principle as combined mode).
If temporal averaging is a general principle, SmoothedAdam should beat AdamW.
If it only helps NS-based optimizers, SmoothedAdam ≈ AdamW.

- Model: MicroGPT (1.2M params)
- Data: TinyShakespeare (char-level)
- All params use Adam (no Muon split). Muon included as reference.

In [ ]:
# Download TinyShakespeare
SHAKES_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(SHAKES_PATH):
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        SHAKES_PATH)
with open(SHAKES_PATH, 'r') as f:
    shakes_text = f.read()
print(f'TinyShakespeare: {len(shakes_text):,} chars')

ds_small = CharDataset(shakes_text, ctx_len=CFG_SMALL['ctx_len'], device=DEVICE)

In [ ]:
CONFIGS_GEN = [
    {'name': 'AdamW (all params)', 'type': 'AdamW_full', 'lr': 1e-3, 'wd': 0.01},
    {'name': 'SmoothedAdam a=0.15', 'type': 'SmoothedAdam_full',
     'lr': 1e-3, 'wd': 0.01, 'smooth_beta': 0.5, 'smooth_alpha': 0.15},
    {'name': 'SmoothedAdam a=0.30', 'type': 'SmoothedAdam_full',
     'lr': 1e-3, 'wd': 0.01, 'smooth_beta': 0.5, 'smooth_alpha': 0.30},
    {'name': 'Muon NS=5 (ref)', 'type': 'Muon'},
]

results_gen = run_all(CONFIGS_GEN, ds_small, CFG_SMALL,
                      total_steps=2000, batch_size=32, warmup_steps=200,
                      base_lr=0.02, eval_every=250)
show_results(results_gen, 'Generalization (Adam vs SmoothedAdam, 1.2M MicroGPT)')